# fix_stats

> Phase 13 statistics: the share of a gap closed (mean paired difference over ceiling minus baseline), the share of a loss recovered, bootstrap intervals over the pairing units, Holm within a family (numpy only).

In [ ]:
#| default_exp fix_stats

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np

from explRL.bench_stats import holm, signFlipP
from explRL.llm_report import NAN, _stat


def bootStat(stat, n, nboot=2000, seed=0):
    "95% percentile interval of stat(idx) over resampled pairing units (nan replicates dropped)."
    if not n:
        return [NAN, NAN]
    v = np.array([stat(i) for i in np.random.default_rng(seed).integers(0, n, size=(nboot, n))])
    v = v[~np.isnan(v)]
    return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] if len(v) else [NAN, NAN]


def bootSE(stat, n, nboot=2000, seed=0):
    "Bootstrap standard error of stat(idx) over resampled pairing units (the SEM's analogue for a ratio; nan replicates dropped)."
    if n < 2:
        return NAN
    v = np.array([stat(i) for i in np.random.default_rng(seed).integers(0, n, size=(nboot, n))])
    v = v[~np.isnan(v)]
    return float(v.std(ddof=1)) if len(v) > 1 else NAN


def _ratio(num, den):
    return float(num / den) if den != 0 else NAN


def _out(f, n, d, b, ceiling):
    return dict(gap=f(np.arange(n)), ci=bootStat(f, n), se=bootSE(f, n), n=n, p=signFlipP(d), diff=_stat(d),
                base=float(b.mean()) if n else NAN, ceiling=ceiling)


def gapClosed(arm, base, ceil):
    """Gap closed = mean paired difference (arm - baseline) / (ceiling - baseline mean), on the pairing units all have.
    arm, base: {unit: value}; ceil: {unit: value} (solo ceiling) or a number (pooled share 1). CI: bootstrap over units."""
    u = sorted(set(arm) & set(base) & (set(ceil) if isinstance(ceil, dict) else set(arm)))
    a, b = np.array([arm[k] for k in u], float), np.array([base[k] for k in u], float)
    c = np.array([ceil[k] for k in u], float) if isinstance(ceil, dict) else np.full(len(u), float(ceil))
    f = lambda i: _ratio((a[i] - b[i]).mean(), c[i].mean() - b[i].mean()) if len(i) else NAN
    return _out(f, len(u), a - b, b, float(c.mean()) if len(u) else NAN)


def lossRecovered(arm, base):
    """Loss recovered = 1 - mean arm loss / mean baseline loss, on the units both have (arm, base: {unit: loss});
    the paired difference and its p-value are baseline loss - arm loss. CI: bootstrap over units."""
    u = sorted(set(arm) & set(base))
    a, b = np.array([arm[k] for k in u], float), np.array([base[k] for k in u], float)
    f = lambda i: 1 - _ratio(a[i].mean(), b[i].mean()) if len(i) else NAN
    return _out(f, len(u), b - a, b, 0.0)


def holmFamily(items, family):
    "Holm-adjusted p over the keys in `family` (marked primary); every other item is a control (pHolm nan)."
    for k, q in zip(family, holm([items[k]['p'] for k in family]) if family else []):
        items[k].update(pHolm=float(q), primary=True)
    for v in items.values():
        v.setdefault('pHolm', NAN)
        v.setdefault('primary', False)
    return items

## Tests

In [ ]:
import math
import numpy as np
from explRL.fix_stats import *

# gap closed against a fixed ceiling: a constant paired difference of 0.3 over a baseline mean of 0.4 closes half the gap
base = {s: 0.4 + 0.02 * (s % 3 - 1) for s in range(10)}
g = gapClosed({s: v + 0.3 for s, v in base.items()}, base, 1.0)
assert abs(g['gap'] - 0.3 / (1 - np.mean(list(base.values())))) < 1e-12 and g['n'] == 10 and g['ceiling'] == 1.0
assert g['ci'][0] <= g['gap'] <= g['ci'][1] and abs(g['diff']['mean'] - 0.3) < 1e-12 and g['p'] < 0.01
# per-unit ceiling (the solo agent on the same episodes), only units all three have; a null arm closes nothing
solo = {s: 0.9 for s in range(8)}
g = gapClosed(base, base, solo)
assert g['gap'] == 0 and g['n'] == 8 and g['p'] == 1.0 and abs(g['ceiling'] - 0.9) < 1e-12
# win rates: baseline wins 2 of 10, the arm 4 of 10 (same episodes), the solo agent 9 of 10 -> 2/7 of the gap
b = {s: float(s < 2) for s in range(10)}
a = {s: float(s < 4) for s in range(10)}
c = {s: float(s < 9) for s in range(10)}
assert abs(gapClosed(a, b, c)['gap'] - 0.2 / 0.7) < 1e-12
# loss recovered: arm loss 0.1 against baseline 0.4 -> 0.75; an equal loss -> 0; no units -> nan
L = lossRecovered({s: 0.1 for s in range(5)}, {s: 0.4 + 0.01 * s for s in range(6)})
assert abs(L['gap'] - (1 - 0.1 / 0.42)) < 1e-12 and L['n'] == 5 and L['diff']['mean'] > 0 and L['ceiling'] == 0.0
assert lossRecovered({1: 0.3}, {1: 0.3})['gap'] == 0 and math.isnan(lossRecovered({}, {1: 1.0})['gap'])
assert all(math.isnan(x) for x in lossRecovered({}, {})['ci'])
# Holm within a family; controls keep nan
items = dict(x=dict(p=0.01), y=dict(p=0.04), z=dict(p=0.001))
holmFamily(items, ['x', 'y'])
assert items['x']['pHolm'] == 0.02 and items['y']['pHolm'] == 0.04 and items['x']['primary']
assert math.isnan(items['z']['pHolm']) and not items['z']['primary']
# the bootstrap is seeded (identical reruns)
assert bootStat(lambda i: float(np.mean(i)), 5) == bootStat(lambda i: float(np.mean(i)), 5)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()